# 🩺 Upgraded Medical Text Classifier (PubMedBERT / BioClinicalBERT)
### SmartSymptom AI — High Accuracy Medical NLP Pipeline
This notebook implements an academic-grade Clinical NLP classification pipeline:
1. **Medical Pretrained Transformer**: Uses  (pretrained on millions of biomedical papers and clinical terms).
2. **Negation & Context Preservation**: Cleans noise while keeping vital clinical phrases (e.g. "no fever", "chest tightness").
3. **Stratified Split**: Rigorous 70/15/15 Train-Validation-Test partitioning.
4. **Top-3 Differential Diagnosis**: Generates calibrated softmax probabilities.
5. **Academic Evaluation**: Generates Confusion Matrix, Macro/Micro F1-Scores, and Classification Report.


In [ ]:
# ==========================================
# 1. SETUP & DEPENDENCIES
# ==========================================
!pip install -q transformers[torch] datasets scikit-learn seaborn matplotlib huggingface_hub

import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"⚡ Running on: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


In [ ]:
# ==========================================
# 2. DATA LOADING & CLINICAL PREPROCESSING
# ==========================================
import os
import glob
import pandas as pd
import re
import joblib
from sklearn.preprocessing import LabelEncoder

# Multi-Source Dataset Loader (Works automatically in Kaggle, Colab, or Local)
df = None

# Method 1: Check if already present in current working directory
if os.path.exists('Symptom2Disease.csv'):
    print("Found local Symptom2Disease.csv")
    df = pd.read_csv('Symptom2Disease.csv')

# Method 2: Check Kaggle /kaggle/input/ directory
if df is None:
    kaggle_paths = glob.glob('/kaggle/input/**/Symptom2Disease.csv', recursive=True) +                    glob.glob('/kaggle/input/**/symptom2disease.csv', recursive=True) +                    glob.glob('/kaggle/input/**/*.csv', recursive=True)
    for p in kaggle_paths:
        if 'symptom' in p.lower():
            print(f"Found dataset in Kaggle input: {p}")
            df = pd.read_csv(p)
            break

# Method 3: Download via kagglehub (Native Kaggle integration)
if df is None:
    try:
        import kagglehub
        print("📥 Downloading dataset via kagglehub...")
        path = kagglehub.dataset_download("niyarrbarman/symptom2disease")
        csv_candidates = glob.glob(os.path.join(path, "*.csv")) + glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
        if csv_candidates:
            df = pd.read_csv(csv_candidates[0])
            print(f"✅ Loaded via kagglehub from: {csv_candidates[0]}")
    except Exception as e:
        print(f"kagglehub notice: {e}")

# Method 4: Download from Hugging Face Datasets
if df is None:
    try:
        from datasets import load_dataset
        print("📥 Loading from Hugging Face Datasets (NeuronZero/Symptom2Disease)...")
        hf_ds = load_dataset("NeuronZero/Symptom2Disease", split="train")
        df = hf_ds.to_pandas()
        print("✅ Loaded successfully from Hugging Face!")
    except Exception as e:
        print(f"Hugging Face notice: {e}")

# Method 5: Google Colab interactive file upload fallback
if df is None:
    try:
        from google.colab import files
        print("⚠️ Please upload your Symptom2Disease.csv file:")
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith('.csv'):
                df = pd.read_csv(fn)
                break
    except Exception as e:
        print(f"Colab upload notice: {e}")

if df is None:
    raise FileNotFoundError(
        "Could not locate 'Symptom2Disease.csv'. "
        "If you are on Kaggle: click '+ Add Input' on the right sidebar, search for 'symptom2disease', and click Add!"
    )

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Normalize column names if needed
if 'label' not in df.columns and 'disease' in df.columns:
    df['label'] = df['disease']
if 'text' not in df.columns and 'symptoms' in df.columns:
    df['text'] = df['symptoms']

print(f"📊 Total samples: {len(df)}")
print(f"🩺 Number of unique diseases: {df['label'].nunique()}")
print(f"Sample distribution:")
print(df['label'].value_counts().head(5))

# Text cleaning preserving clinical negation and context
def clean_medical_text(text):
    text = str(text).lower()
    # Normalize whitespaces
    text = re.sub(r'\s+', ' ', text)
    # Remove unwanted special characters but preserve punctuation relevant to sentence pauses
    text = re.sub(r'[^a-zA-Z0-9\s,\.\-]', '', text)
    return text.strip()

df['cleaned_text'] = df['text'].apply(clean_medical_text)

# Encode disease labels
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])
num_classes = len(le.classes_)

# Save Label Encoder
os.makedirs('saved_model', exist_ok=True)
joblib.dump(le, 'saved_model/label_encoder.pkl')
print(f"✅ {num_classes} classes encoded and saved.")


In [ ]:
# ==========================================
# 3. STRATIFIED TRAIN / VAL / TEST SPLIT
# ==========================================
# 70% Train, 15% Validation, 15% Independent Test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    df['cleaned_text'].tolist(),
    df['label_id'].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=df['label_id']
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.1765, # 0.1765 * 0.85 approx 0.15 of total
    random_state=42,
    stratify=y_train_val
)

print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")


In [ ]:
# ==========================================
# 4. TOKENIZATION & MODEL INITIALIZATION
# ==========================================
# We use PubMedBERT (trained on PubMed biomedical abstracts) for superior clinical domain understanding
# Fallback to 'distilbert-base-uncased' if lightweight inference is required
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
# MODEL_NAME = "distilbert-base-uncased"  # Alternative lightweight option

print(f"📦 Loading tokenizer and base model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=num_classes
).to(device)

class ClinicalDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ClinicalDataset(X_train, y_train, tokenizer)
val_dataset = ClinicalDataset(X_val, y_val, tokenizer)
test_dataset = ClinicalDataset(X_test, y_test, tokenizer)


In [ ]:
# ==========================================
# 5. TRAINING WITH EARLY STOPPING & EVALUATION
# ==========================================
import inspect

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'macro_f1': macro_f1}

# Robust arguments compatible with all versions of transformers in Kaggle / Colab
sig = inspect.signature(TrainingArguments.__init__).parameters
strategy_key = 'eval_strategy' if 'eval_strategy' in sig else 'evaluation_strategy'

base_args = {
    'output_dir': './checkpoints',
    strategy_key: 'epoch',
    'save_strategy': 'epoch',
    'learning_rate': 2e-5,
    'per_device_train_batch_size': 16,
    'per_device_eval_batch_size': 32,
    'num_train_epochs': 8,
    'weight_decay': 0.01,
    'warmup_steps': 40,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'macro_f1',
    'greater_is_better': True,
    'logging_steps': 20,
    'report_to': 'none',
    'fp16': torch.cuda.is_available()
}

# Keep only arguments supported by the installed transformers version
compatible_args = {k: v for k, v in base_args.items() if k in sig}
training_args = TrainingArguments(**compatible_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 Training model...")
trainer.train()

# Save final best model and tokenizer
model.save_pretrained('saved_model/best_medical_text_model')
tokenizer.save_pretrained('saved_model/best_medical_text_model')
print("✅ Best model saved to saved_model/best_medical_text_model")


In [ ]:
# ==========================================
# 6. INDEPENDENT TEST EVALUATION & CONFUSION MATRIX
# ==========================================
print("🧪 Evaluating on Independent Test Set...")
test_results = trainer.predict(test_dataset)
y_pred = np.argmax(test_results.predictions, axis=-1)
y_true = np.array(y_test)

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')

print(f"
🎯 Final Test Accuracy: {acc*100:.2f}%")
print(f"🎯 Final Test Macro F1: {macro_f1*100:.2f}%")
print("
--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=le.classes_))

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Test Set Confusion Matrix — Medical Text Classifier', fontsize=16)
plt.xlabel('Predicted Disease', fontsize=12)
plt.ylabel('True Disease', fontsize=12)
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig('saved_model/confusion_matrix.png', dpi=300)
plt.show()
print("✅ Confusion matrix saved to saved_model/confusion_matrix.png")


In [ ]:
# ==========================================
# 7. INFERENCE FUNCTION WITH TOP-3 DIFFERENTIAL DIAGNOSIS
# ==========================================
def predict_symptoms(text, top_k=3):
    cleaned = clean_medical_text(text)
    inputs = tokenizer(cleaned, return_tensors='pt', truncation=True, padding=True, max_length=128).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    
    top_indices = np.argsort(probs)[::-1][:top_k]
    results = [
        {
            'disease': le.inverse_transform([idx])[0],
            'confidence': float(probs[idx]),
            'percentage': f"{probs[idx]*100:.1f}%"
        }
        for idx in top_indices
    ]
    return {
        'input': text,
        'primary_diagnosis': results[0]['disease'],
        'confidence': results[0]['confidence'],
        'top_candidates': results
    }

# Test Examples
examples = [
    "I have a dry cough, shortness of breath, high fever, and sudden loss of smell and taste.",
    "Severe itching on the skin, red ring-shaped rash with raised scaly edges on the arm.",
    "Sharp stabbing chest pain radiating to my left arm, cold sweat, and nausea."
]

for ex in examples:
    pred = predict_symptoms(ex)
    print(f"
Input: '{pred['input']}'")
    print(f"Top Diagnosis: {pred['primary_diagnosis']} ({pred['top_candidates'][0]['percentage']})")
    print(f"Top 3: {[c['disease'] + ' (' + c['percentage'] + ')' for c in pred['top_candidates']]}")
